# Browser ID ↔ IP Address — Deanonymization Graph

Interactive network graph built from `device_usage_logs.jsonl`.

- **Blue nodes** = Browser IDs (per-device, per-site stable cookie)
- **Green nodes** = IP addresses
- **Edges** = a browser ID was seen on that IP (thickness = visit count)
- **Node size** = number of connections (degree)

Browser IDs that share the same IP node likely belong to the same physical device or user.

In [ ]:
import polars as pl
import networkx as nx
import plotly.graph_objects as go

## Load data

In [ ]:
df = pl.read_ndjson("internet_logs/device_usage_logs.jsonl")

missing = [c for c in ("ip_address", "browser_id") if c not in df.columns]
if missing:
    raise RuntimeError(
        f"Log file is missing columns: {missing}\n"
        "The existing logs were generated before ip_address/browser_id were added.\n"
        "Delete internet_logs/device_usage_logs.jsonl and re-run the simulation to get fresh data."
    )

df = df.filter(pl.col("ip_address").is_not_null() & pl.col("browser_id").is_not_null())

print(f"Usable rows:        {len(df):,}")
print(f"Unique browser IDs: {df['browser_id'].n_unique()}")
print(f"Unique IPs:         {df['ip_address'].n_unique()}")
df.head(3)

## Build graph

In [ ]:
G = nx.Graph()

# Count how many times each (browser_id, ip_address) pair co-occurred
edge_counts = (
    df.group_by(["browser_id", "ip_address"])
    .agg(pl.len().alias("count"))
)

# Build a lookup: browser_id -> (device_id, sites visited)
bid_meta = (
    df.group_by("browser_id")
    .agg([
        pl.col("device_id").first(),
        pl.col("website").unique().sort().str.join(", ").alias("sites"),
    ])
    .to_dicts()
)
bid_info = {r["browser_id"]: r for r in bid_meta}

# Step 1: add all nodes and edges (hover filled in after)
for row in edge_counts.iter_rows(named=True):
    bid_full = row["browser_id"]
    ip       = row["ip_address"]
    bid_node = f"id:{bid_full[:8]}"
    ip_node  = f"ip:{ip}"
    G.add_node(bid_node, node_type="browser_id", full_id=bid_full)
    G.add_node(ip_node,  node_type="ip")
    G.add_edge(bid_node, ip_node, weight=row["count"])

# Step 2: set hover text now that degrees are final
for node, data in G.nodes(data=True):
    if data["node_type"] == "browser_id":
        meta = bid_info.get(data["full_id"], {})
        data["hover"] = (
            f"<b>{data['full_id']}</b><br>"
            f"Device: {meta.get('device_id', '?')}<br>"
            f"Sites: {meta.get('sites', '?')}<br>"
            f"Connections: {G.degree[node]}"
        )
    else:
        data["hover"] = f"<b>{node}</b><br>Connected browser IDs: {G.degree[node]}"

print(f"Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}")

## Render interactive graph

In [ ]:
import webbrowser, os, math

# --- Cluster layout: IPs in a circle, browser_ids orbiting their IP ---
def cluster_layout(G, ip_ring_radius=3.0, bid_orbit_radius=0.5):
    ip_nodes  = [n for n, d in G.nodes(data=True) if d.get("node_type") == "ip"]
    pos = {}

    n_ips = len(ip_nodes)
    for i, ip in enumerate(ip_nodes):
        # Space IP nodes evenly on a large circle
        angle = 2 * math.pi * i / n_ips
        cx = ip_ring_radius * math.cos(angle)
        cy = ip_ring_radius * math.sin(angle)
        pos[ip] = (cx, cy)

        # Orbit browser_id neighbours around this IP
        neighbours = list(G.neighbors(ip))
        n_nb = len(neighbours)
        for j, bid in enumerate(neighbours):
            if bid not in pos:   # a browser_id is only connected to one IP
                a = 2 * math.pi * j / max(n_nb, 1)
                pos[bid] = (cx + bid_orbit_radius * math.cos(a),
                            cy + bid_orbit_radius * math.sin(a))
    return pos

pos = cluster_layout(G)

# --- Edge traces ---
edge_traces = []
for u, v, data in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode="lines",
        line=dict(width=max(0.5, data.get("weight", 1) * 1.2), color="#cccccc"),
        hoverinfo="none",
        showlegend=False,
    ))

# --- Node traces ---
node_traces = []
for node_type, color, label in [
    ("ip",         "#2ecc71", "IP address"),
    ("browser_id", "#3498db", "Browser ID"),
]:
    nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == node_type]
    if not nodes:
        continue
    node_traces.append(go.Scatter(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        mode="markers+text",
        name=label,
        text=nodes,
        textposition="top center",
        textfont=dict(size=8, color="#333"),
        hovertext=[G.nodes[n].get("hover", n) for n in nodes],
        hoverinfo="text",
        marker=dict(
            # IP nodes are larger hubs; browser_id nodes scale with degree
            size=[20 if node_type == "ip" else 8 + 3 * G.degree[n] for n in nodes],
            color=color,
            line=dict(width=1.5, color="white"),
            symbol="circle" if node_type == "browser_id" else "diamond",
        ),
    ))

fig = go.Figure(
    data=edge_traces + node_traces,
    layout=go.Layout(
        title=dict(text="Browser ID ↔ IP Address — Deanonymization Graph", font=dict(size=18)),
        showlegend=True,
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)", bordercolor="#ccc", borderwidth=1),
        hovermode="closest",
        height=850,
        margin=dict(l=20, r=20, t=60, b=20),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x"),
        plot_bgcolor="#fafafa",
    )
)

out = os.path.abspath("internet_logs/graph.html")
fig.write_html(out)
webbrowser.open(f"file://{out}")
print(f"Graph saved and opened: {out}")